# Weather Prediction System

Predicts next-day maximum temperature using historical weather data and a **Ridge Regression** model. Features are progressively engineered — rolling averages, monthly means, and day-of-year averages — to improve accuracy.

## 1. Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('local_weather.csv', index_col='DATE')
df.head()


## 2. Null Value Check

In [ ]:
df.apply(pd.isnull).sum() / df.shape[0]


## 3. Select & Rename Relevant Columns

In [ ]:
data = df[['PRCP', 'SNOW', 'SNWD', 'TMAX', 'TMIN']].copy()
data.columns = ['precip', 'snow', 'snow_depth', 'temp_max', 'temp_min']
data.apply(pd.isnull).sum()


## 4. Drop Low-Signal Columns

Both `snow` and `snow_depth` are overwhelmingly zero — they add noise rather than signal.

In [ ]:
print(data['snow'].value_counts())
print(data['snow_depth'].value_counts())


In [ ]:
data.drop(columns=['snow', 'snow_depth'], inplace=True)


## 5. Handle Missing Values

In [ ]:
# rows where precip is null
data[pd.isnull(data['precip'])]


In [ ]:
# check a specific date
data.loc['2013-12-15', :]


In [ ]:
# precip is mostly 0 — safe to fill nulls with 0
print(data['precip'].value_counts() / data.shape[0])
data['precip'] = data['precip'].fillna(0)


In [ ]:
# inspect temp_min nulls
data[pd.isnull(data['temp_min'])]


In [ ]:
# view surrounding rows for context
data.loc['2011-12-18':'2011-12-28']


In [ ]:
# forward-fill remaining nulls
data = data.ffill()
data.apply(pd.isnull).sum()


## 6. Data Quality & Type Checks

In [ ]:
# check for sentinel 9999 values from data documentation
data.apply(lambda col: (col == 9999).sum())


In [ ]:
# convert index to datetime
data.index = pd.to_datetime(data.index)
print(data.dtypes)
data.index


## 7. Exploratory Visualisation

In [ ]:
data[['temp_max', 'temp_min']].plot(figsize=(12, 4), title='Daily Temperature Range')


In [ ]:
data.index.year.value_counts().sort_index()


In [ ]:
data['precip'].plot(figsize=(12, 3), title='Daily Precipitation')


In [ ]:
data.groupby(data.index.year)['precip'].sum().plot(
    kind='bar', figsize=(12, 4), title='Annual Total Precipitation'
)


## 8. Create Target Variable

The target is **tomorrow's** `temp_max` — we shift by -1 so each row's label is the next day's value.

In [ ]:
data['target'] = data['temp_max'].shift(-1)
# drop last row — it has no valid target
data = data.iloc[:-1, :].copy()
data.tail()


## 9. Baseline Ridge Regression Model

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

model = Ridge(alpha=0.1)

base_features = ['precip', 'temp_max', 'temp_min']

train = data.loc[:'2020-12-31']
test  = data.loc['2021-01-01':]

model.fit(train[base_features], train['target'])
preds = model.predict(test[base_features])

baseline_mse = mean_squared_error(test['target'], preds)
print(f'Baseline MSE: {baseline_mse:.4f}')


In [ ]:
results = pd.DataFrame({'actual': test['target'], 'predicted': preds}, index=test.index)
results.plot(figsize=(12, 4), title='Actual vs Predicted — Baseline')


In [ ]:
print('Feature coefficients:', model.coef_)


## 10. Feature Engineering — Rolling & Ratio Features

In [ ]:
data['rolling_30d_max']   = data['temp_max'].rolling(30).mean()
data['ratio_rolling_day'] = data['rolling_30d_max'] / data['temp_max']
data['ratio_max_min']     = data['temp_max'] / data['temp_min']

# drop the first 30 rows — rolling window produces NaN there
data = data.iloc[30:, :].copy()


## 11. Prediction Helper Function

In [ ]:
def run_model(features, df, ridge_model):
    tr = df.loc[:'2020-12-31']
    te = df.loc['2021-01-01':]

    ridge_model.fit(tr[features], tr['target'])
    preds = ridge_model.predict(te[features])

    mse = mean_squared_error(te['target'], preds)
    out = pd.DataFrame({'actual': te['target'], 'predicted': preds}, index=te.index)
    return mse, out


## 12. Improved Model — With Engineered Features

In [ ]:
features_v2 = ['precip', 'temp_max', 'temp_min', 'ratio_rolling_day', 'ratio_max_min']

mse_v2, results_v2 = run_model(features_v2, data, model)
print(f'Improved MSE: {mse_v2:.4f}')
results_v2.plot(figsize=(12, 4), title='Actual vs Predicted — Engineered Features')


## 13. Add Historical Averages

In [ ]:
data['monthly_avg'] = (
    data['temp_max']
    .groupby(data.index.month)
    .transform(lambda x: x.expanding(1).mean())
)
data['day_of_year_avg'] = (
    data['temp_max']
    .groupby(data.index.day_of_year)
    .transform(lambda x: x.expanding(1).mean())
)


In [ ]:
features_v3 = features_v2 + ['monthly_avg', 'day_of_year_avg']

mse_v3, results_v3 = run_model(features_v3, data, model)
print(f'Final MSE: {mse_v3:.4f}')
results_v3.plot(figsize=(12, 4), title='Actual vs Predicted — Full Feature Set')


## 14. Model Diagnostics

In [ ]:
print('Feature correlations with target:')
print(data.corr()['target'].sort_values(ascending=False))


In [ ]:
results_v3['error'] = (results_v3['actual'] - results_v3['predicted']).abs()
print('Worst predictions:')
results_v3.sort_values('error', ascending=False).head(10)
